# Morphological Measurements Analysis

This notebook analyzes morphological measurements data from patients examined at different time points. The analysis focuses on:
1. Comparing groups (ОГ vs КГ) across examination phases for location "средняя нос.раковина"
2. Comparing groups at the "день операции" phase

## Data Structure
- **id**: sample ID
- **группа**: group name (ОГ, КГ)
- **локация**: location (полип, средняя нос.раковина)
- **этап**: examination phases (день операции, 6-7 сутки, 1-3 мес., 1 год)
- **поле**: field (1-5)

## Measurements
The analysis covers the following morphological measurements:
- круглоклеточная воспалительная инфильтрация
- бокаловидные клетки
- отек реснички эпителия
- фиброз
- гиперплазия респираторного эпителия
- плоскоклеточная метаплазия
- эозинофилы
- нейтрофилы

## 1. Load Required Libraries

Loading necessary R libraries for data manipulation, visualization, and statistical analysis.

In [ ]:
# Load required libraries
library(dplyr) # Data manipulation
library(ggplot2) # Data visualization
library(tidyr) # Data reshaping
library(readxl) # Reading Excel files
library(readr) # Reading CSV files
library(broom) # Tidy statistical output
library(car) # Companion to Applied Regression
library(gridExtra) # Arranging plots
library(knitr) # Dynamic reports
library(reshape2) # Data reshaping
library(RColorBrewer) # Color palettes

# Set theme for plots
theme_set(theme_minimal())

print("Libraries loaded successfully!")

## 2. Import and Examine Data

Loading the morphological measurements dataset and examining its structure.

In [ ]:
# Load the data from mucous20240825.xlsx
data <- read_excel("C:\\Analysis\\OTOLARING\\Nidelko\\mucous20240825.xlsx", sheet = "данные")

# Display basic information about the loaded data
print(paste("Data dimensions:", nrow(data), "rows,", ncol(data), "columns"))
print("\nColumn names:")
print(colnames(data))
print("\nData structure:")
str(data)

In [ ]:
data <- data %>%
  select(
    "группа", "id", "локация", "этап", "поле", "круглоклеточная воспалительная инфильтрация", "бокаловидные клетки",
    "отек", "реснички эпителия", "фиброз", "гиперплазия респираторного эпителия", "плоскоклеточная метаплазия",
    "эозинофилы", "нейтрофилы"
  )

## 3. Data Preprocessing and Cleaning

Preparing the dataset for analysis by ensuring proper factor levels and data types.

In [ ]:
# Data preprocessing and cleaning
print("Original column names:")
print(colnames(data))

# Check if we need to identify the correct column names
# Look for columns that might contain our key variables
possible_id_cols <- colnames(data)[grepl("id|ID|Id", colnames(data), ignore.case = TRUE)]
possible_group_cols <- colnames(data)[grepl("группа|group|Group", colnames(data), ignore.case = TRUE)]
possible_location_cols <- colnames(data)[grepl("локация|location|Location", colnames(data), ignore.case = TRUE)]
possible_phase_cols <- colnames(data)[grepl("этап|phase|Phase|stage|Stage", colnames(data), ignore.case = TRUE)]
possible_field_cols <- colnames(data)[grepl("поле|field|Field", colnames(data), ignore.case = TRUE)]

print("\nPossible column matches:")
print(paste("ID columns:", paste(possible_id_cols, collapse = ", ")))
print(paste("Group columns:", paste(possible_group_cols, collapse = ", ")))
print(paste("Location columns:", paste(possible_location_cols, collapse = ", ")))
print(paste("Phase columns:", paste(possible_phase_cols, collapse = ", ")))
print(paste("Field columns:", paste(possible_field_cols, collapse = ", ")))

# Clean the data - this will need to be adjusted based on actual column names
data_clean <- data %>%
  # Remove any completely empty rows
  filter(!if_all(everything(), is.na)) %>%
  # Convert character variables to factors with proper levels
  mutate(
    # You may need to adjust these column references based on your actual data
    across(where(is.character), as.factor)
  )

# Display unique values for key categorical variables
print("\nUnique values in categorical columns:")
categorical_cols <- sapply(data_clean, is.factor)
for (col in names(data_clean)[categorical_cols]) {
  cat("\n", col, ":\n")
  print(table(data_clean[[col]], useNA = "ifany"))
}

# Check for missing values
print("\nMissing values summary:")
missing_summary <- sapply(data_clean, function(x) sum(is.na(x)))
print(missing_summary[missing_summary > 0])

print("\nCleaned data dimensions:")
print(dim(data_clean))

## 4. Exploratory Data Analysis

Initial exploration of the data to understand patterns and distributions.

In [ ]:
# Exploratory Data Analysis

# Overall summary statistics by group
print("Summary statistics by group:")
data_clean %>%
  group_by(группа) %>%
  summarise(
    n = n(),
    mean_value = mean(measurement_value, na.rm = TRUE),
    median_value = median(measurement_value, na.rm = TRUE),
    sd_value = sd(measurement_value, na.rm = TRUE),
    min_value = min(measurement_value, na.rm = TRUE),
    max_value = max(measurement_value, na.rm = TRUE)
  ) %>%
  kable(digits = 2)

# Data distribution by location and phase
print("\nData distribution by location and phase:")
data_clean %>%
  group_by(локация, этап, группа) %>%
  summarise(
    n = n(),
    mean_value = mean(measurement_value, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  pivot_wider(names_from = группа, values_from = c(n, mean_value)) %>%
  kable(digits = 2)

# Check sample sizes for main comparisons
print("\nSample sizes for средняя нос.раковина by phase and group:")
data_clean %>%
  filter(локация == "средняя нос.раковина") %>%
  group_by(этап, группа) %>%
  summarise(n = n(), .groups = "drop") %>%
  pivot_wider(names_from = группа, values_from = n) %>%
  kable()

print("\nSample sizes for день операции by location and group:")
data_clean %>%
  filter(этап == "день операции") %>%
  group_by(локация, группа) %>%
  summarise(n = n(), .groups = "drop") %>%
  pivot_wider(names_from = группа, values_from = n) %>%
  kable()

## 5. Group Comparison by Phase for средняя нос.раковина

Comparing groups (ОГ vs КГ) across all examination phases for location "средняя нос.раковина".

In [ ]:
# Analysis 1: Compare groups on each phase for локация = средняя нос.раковина

# Filter data for средняя нос.раковина
nasal_data <- data_clean %>%
  filter(локация == "средняя нос.раковина")

print("Analysis 1: Group comparisons by phase for средняя нос.раковина")
print("============================================================")

# Function to perform statistical test (t-test or Mann-Whitney U test)
perform_group_test <- function(data_subset, test_name) {
  og_data <- data_subset %>%
    filter(группа == "ОГ") %>%
    pull(measurement_value)
  kg_data <- data_subset %>%
    filter(группа == "КГ") %>%
    pull(measurement_value)

  # Check if we have enough data
  if (length(og_data) < 3 || length(kg_data) < 3) {
    return(data.frame(
      test = test_name,
      p_value = NA,
      method = "Insufficient data",
      og_mean = mean(og_data, na.rm = TRUE),
      kg_mean = mean(kg_data, na.rm = TRUE),
      og_n = length(og_data),
      kg_n = length(kg_data)
    ))
  }

  # Check normality with Shapiro-Wilk test
  og_normal <- ifelse(length(og_data) > 2, shapiro.test(og_data)$p.value > 0.05, FALSE)
  kg_normal <- ifelse(length(kg_data) > 2, shapiro.test(kg_data)$p.value > 0.05, FALSE)

  # Choose appropriate test
  if (og_normal && kg_normal) {
    # Use t-test for normal data
    test_result <- t.test(og_data, kg_data)
    method <- "t-test"
  } else {
    # Use Mann-Whitney U test for non-normal data
    test_result <- wilcox.test(og_data, kg_data)
    method <- "Mann-Whitney U"
  }

  return(data.frame(
    test = test_name,
    p_value = test_result$p.value,
    method = method,
    og_mean = mean(og_data, na.rm = TRUE),
    kg_mean = mean(kg_data, na.rm = TRUE),
    og_n = length(og_data),
    kg_n = length(kg_data)
  ))
}

# Perform tests for each phase
phase_results <- data.frame()

for (phase in levels(nasal_data$этап)) {
  phase_data <- nasal_data %>% filter(этап == phase)

  if (nrow(phase_data) > 0) {
    result <- perform_group_test(phase_data, paste("Phase:", phase))
    phase_results <- rbind(phase_results, result)
  }
}

print("Results for средняя нос.раковина by phase:")
print(phase_results)
kable(phase_results, digits = 4)

## 6. Group Comparison on день операции

Comparing groups across different measurements and locations for examination phase "день операции".

In [ ]:
# Analysis 2: Compare groups on этап = день операции

# Filter data for день операции
surgery_day_data <- data_clean %>%
  filter(этап == "день операции")

print("Analysis 2: Group comparisons for день операции")
print("===============================================")

# Perform tests for each location on surgery day
location_results <- data.frame()

for (location in levels(surgery_day_data$локация)) {
  location_data <- surgery_day_data %>% filter(локация == location)

  if (nrow(location_data) > 0) {
    result <- perform_group_test(location_data, paste("Location:", location))
    location_results <- rbind(location_results, result)
  }
}

print("Results for день операции by location:")
print(location_results)
kable(location_results, digits = 4)

# Additional analysis: Compare groups by measurement type on surgery day
print("\nDetailed analysis by measurement type on день операции:")
measurement_results <- data.frame()

for (measurement in levels(surgery_day_data$measurement_name)) {
  measurement_data <- surgery_day_data %>% filter(measurement_name == measurement)

  if (nrow(measurement_data) > 0) {
    result <- perform_group_test(measurement_data, paste("Measurement:", measurement))
    measurement_results <- rbind(measurement_results, result)
  }
}

print(measurement_results)
kable(measurement_results, digits = 4)

## 7. Statistical Testing and Results Summary

Comprehensive summary of all statistical tests with effect sizes and significance levels.

In [ ]:
# Statistical Results Summary

# Combine all results
all_results <- rbind(
  phase_results %>% mutate(analysis = "Phase comparison (средняя нос.раковина)"),
  location_results %>% mutate(analysis = "Location comparison (день операции)"),
  measurement_results %>% mutate(analysis = "Measurement comparison (день операции)")
)

# Add significance levels
all_results <- all_results %>%
  mutate(
    significance = case_when(
      is.na(p_value) ~ "N/A",
      p_value < 0.001 ~ "***",
      p_value < 0.01 ~ "**",
      p_value < 0.05 ~ "*",
      p_value < 0.1 ~ ".",
      TRUE ~ ""
    ),
    effect_size = abs(og_mean - kg_mean),
    effect_direction = case_when(
      is.na(og_mean) | is.na(kg_mean) ~ "N/A",
      og_mean > kg_mean ~ "ОГ > КГ",
      og_mean < kg_mean ~ "ОГ < КГ",
      TRUE ~ "ОГ = КГ"
    )
  )

print("COMPREHENSIVE RESULTS SUMMARY")
print("============================")
print("Significance codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1")
print("")

# Display results table
kable(
  all_results %>%
    select(
      analysis, test, method, og_mean, kg_mean, og_n, kg_n,
      p_value, significance, effect_size, effect_direction
    ),
  digits = 4,
  col.names = c(
    "Analysis", "Test", "Method", "ОГ Mean", "КГ Mean",
    "ОГ N", "КГ N", "P-value", "Sig.", "Effect Size", "Direction"
  )
)

# Summary of significant results
significant_results <- all_results %>%
  filter(!is.na(p_value) & p_value < 0.05)

print(paste("\nNumber of significant results (p < 0.05):", nrow(significant_results)))

if (nrow(significant_results) > 0) {
  print("\nSignificant results:")
  kable(
    significant_results %>%
      select(analysis, test, p_value, effect_direction),
    digits = 4
  )
}

## 8. Data Visualization

Creating comprehensive visualizations to illustrate group differences across phases and measurements.

In [ ]:
# Data Visualization

# Plot 1: Group comparison by phase for средняя нос.раковина
p1 <- nasal_data %>%
  ggplot(aes(x = этап, y = measurement_value, fill = группа)) +
  geom_boxplot(alpha = 0.7) +
  geom_point(
    position = position_jitterdodge(jitter.width = 0.2),
    alpha = 0.5, size = 0.8
  ) +
  scale_fill_manual(values = c("КГ" = "#E69F00", "ОГ" = "#56B4E9")) +
  labs(
    title = "Group Comparison by Phase (средняя нос.раковина)",
    x = "Examination Phase",
    y = "Measurement Value",
    fill = "Group"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

print(p1)

# Plot 2: Group comparison for день операции by location
p2 <- surgery_day_data %>%
  ggplot(aes(x = локация, y = measurement_value, fill = группа)) +
  geom_boxplot(alpha = 0.7) +
  geom_point(
    position = position_jitterdodge(jitter.width = 0.2),
    alpha = 0.5, size = 0.8
  ) +
  scale_fill_manual(values = c("КГ" = "#E69F00", "ОГ" = "#56B4E9")) +
  labs(
    title = "Group Comparison by Location (день операции)",
    x = "Location",
    y = "Measurement Value",
    fill = "Group"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

print(p2)

In [ ]:
# Plot 3: Heatmap of mean values by group, phase, and location
heatmap_data <- data_clean %>%
  group_by(группа, этап, локация) %>%
  summarise(
    mean_value = mean(measurement_value, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  unite("phase_location", этап, локация, sep = " - ")

p3 <- heatmap_data %>%
  ggplot(aes(x = phase_location, y = группа, fill = mean_value)) +
  geom_tile(color = "white", size = 0.5) +
  scale_fill_gradient2(
    low = "blue", mid = "white", high = "red",
    midpoint = median(heatmap_data$mean_value, na.rm = TRUE),
    name = "Mean Value"
  ) +
  labs(
    title = "Heatmap: Mean Values by Group, Phase, and Location",
    x = "Phase - Location",
    y = "Group"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

print(p3)

# Plot 4: Measurement type comparison for день операции
p4 <- surgery_day_data %>%
  ggplot(aes(x = measurement_name, y = measurement_value, fill = группа)) +
  geom_boxplot(alpha = 0.7) +
  scale_fill_manual(values = c("КГ" = "#E69F00", "ОГ" = "#56B4E9")) +
  labs(
    title = "Measurement Type Comparison (день операции)",
    x = "Measurement Type",
    y = "Measurement Value",
    fill = "Group"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  coord_flip() # Flip coordinates for better readability

print(p4)

## Summary and Conclusions

This analysis provides a comprehensive comparison of morphological measurements between groups ОГ and КГ across different examination phases and locations.

### Key Findings:

1. **Phase Analysis for средняя нос.раковина**: 
   - Statistical comparisons were performed between groups ОГ and КГ for each examination phase
   - Appropriate statistical tests were automatically selected based on data normality

2. **Surgery Day Analysis**: 
   - Group comparisons were conducted for the день операции phase
   - Analysis included comparisons by location and measurement type

3. **Statistical Methods Used**:
   - T-tests for normally distributed data
   - Mann-Whitney U tests for non-normally distributed data
   - Significance levels: p < 0.05 for statistical significance

### Next Steps:
1. Replace the sample data with your actual morphological measurements dataset
2. Adjust file path in the data loading section
3. Modify measurement names and categories as needed
4. Consider additional analyses such as longitudinal modeling or repeated measures ANOVA
5. Add multiple comparison corrections if conducting many simultaneous tests

### Code Adaptations Needed:
- Update data loading path and format
- Ensure column names match your dataset
- Modify factor levels to match your specific categories
- Add additional measurement types if needed